# 02 · Lateral-growth clusters

Repeat passes of the same RGT are offset laterally, so beams are grouped into **clusters** of passes over the same piece of coast.

For each family, beams are ordered across-track by their position at a common offshore distance `XM` (500 m). Starting from one outer beam (the reference), the cluster grows inward. A candidate beam is added when:

- its profile brackets `X0` (500 m) and `|elev(X0) − elev_ref(X0)| ≤ bias_tolerance`,
- its date is not already in the cluster (same-date beams are skipped, not breakpoints),
- the cluster stays within `SIZE_LIMIT_M` (180 m) of lateral width.

A beam that fails the bias test or the width limit starts the next cluster. Growth runs from both sides; duplicate beam sets are then removed. Clusters need at least `MIN_PROFILES_PER_CLUSTER` (2) beams.

Requires the package installed from the repository root (`pip install -e .`).
Paths come from `configs/north_slope.toml`; all parameters and their defaults are in `src/is2retreat/config.py`.

In [ ]:
TRACK_ID = "0129"
CONFIG = "../configs/north_slope.toml"
SOURCE = "auto"
BIAS_TOLERANCE = 0.5   # m

In [ ]:
import matplotlib.pyplot as plt

from is2retreat import load_config, prepare_track_inputs
from is2retreat.angles import compute_cluster_angles
from is2retreat.clustering import add_cluster_width_m, make_clusters, select_clusters_per_family
from is2retreat.diagnostics import build_cluster_beam_elevation_check

paths, params = load_config(CONFIG)
inputs = prepare_track_inputs(TRACK_ID, paths, params, source=SOURCE, verbose=False)
dataset_raw, shoreline_gdf, utm_epsg = inputs.dataset_raw, inputs.shoreline_gdf, inputs.utm_epsg
print(f"Track {inputs.track_id}: {dataset_raw['beam_id'].nunique()} beams after preprocessing (notebook 01)")

## Grow clusters

In [ ]:
clusters_gdf, beam_lines = make_clusters(
    dataset_raw,
    utm_epsg=utm_epsg,
    bias_tolerance=BIAS_TOLERANCE,
    xm=params.XM,
    x0=params.X0,
    track_id=inputs.track_id,
    min_beams=params.MIN_PROFILES_PER_CLUSTER,
    size_limit=params.SIZE_LIMIT_M,
)
clusters_gdf[["gt_family", "cluster_id", "growth_side", "reference_beam", "num_beams", "break_reason"]]

## Beam/shoreline crossing angles

Angle between each beam and the local shoreline tangent (0–90°). The GIE correction (notebook 03) uses it.

In [ ]:
clusters_gdf, beam_angle_table = compute_cluster_angles(
    clusters_gdf, shoreline_gdf, dataset_raw,
    track_id=inputs.track_id, bias_tolerance=BIAS_TOLERANCE,
    mode_bin_width=params.ANGLE_MODE_BIN_WIDTH,
    shoreline_search_radius=params.ANGLE_SEARCH_RADIUS,
)
beam_angle_table.head()

## Select clusters and measure their width

In [ ]:
selected, skipped, selection_summary = select_clusters_per_family(
    clusters_gdf, min_profiles=params.MIN_PROFILES_PER_CLUSTER, track_id=inputs.track_id
)
selected = add_cluster_width_m(selected, dataset_raw, xm=params.XM, utm_epsg=utm_epsg)
selection_summary

In [ ]:
selected[["gt_family", "cluster_id", "growth_side", "num_beams", "cluster_width_m",
          "angle_median_deg", "break_reason"]].sort_values(["gt_family", "cluster_id"])

## Elevation at X0 of every member beam

In [ ]:
check = build_cluster_beam_elevation_check(dataset_raw, selected, x0=params.X0)
check.head(20)

## Map

In [ ]:
fams = sorted(selected["gt_family"].unique())
fig, axes = plt.subplots(1, len(fams), figsize=(5 * len(fams), 5), squeeze=False)
shoreline_utm = shoreline_gdf.to_crs(utm_epsg)

for ax, fam in zip(axes[0], fams):
    sel = selected[selected["gt_family"] == fam]
    x0_, y0_, x1_, y1_ = sel.total_bounds
    pad = 300
    shoreline_utm.cx[x0_ - pad:x1_ + pad, y0_ - pad:y1_ + pad].plot(ax=ax, color="k", linewidth=1)
    beam_lines[beam_lines["gt_family"] == fam].plot(ax=ax, color="lightgray", linewidth=0.6)
    sel.plot(ax=ax, column="cluster_id", categorical=True, alpha=0.35, edgecolor="k", linewidth=0.5)
    for _, row in sel.iterrows():
        c = row.geometry.centroid
        ax.annotate(str(row["cluster_id"]), (c.x, c.y), fontsize=8, ha="center")
    ax.set_title(f"{fam}: {len(sel)} clusters (bias tol {BIAS_TOLERANCE} m)")
    ax.set_xlim(x0_ - pad, x1_ + pad)
    ax.set_ylim(y0_ - pad, y1_ + pad)
    ax.set_aspect("equal")
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()